# Phase I — DINOv2 + FPN road masks

Self-supervised ViT backbone ([DINOv2](https://arxiv.org/abs/2304.07193) · [paper PDF](papers/DINOv2_LearningRobustVisualFeatures.pdf)) with a light FPN decoder for dense road masks.

Goal: better continuity under canopy / occlusion (esp. tile `493626`) vs DeepLab / SegFormer.

Outputs: `best_road_model_dinov2_fpn.pth`, `outputs/masks_dinov2/`


## Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Install packages


In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers", "accelerate", "albumentations", "opencv-python", "tqdm",
    "segmentation-models-pytorch",
], check=True)
print("ok")


## Paths & split


In [ ]:
import os, random

DRIVE_BASE = "/content/drive/MyDrive/RouteResilience"
SAVE_DIR = f"{DRIVE_BASE}/checkpoints"
DATA_DIR = f"{DRIVE_BASE}/datasets/train"
MASK_DIR = f"{DRIVE_BASE}/outputs/masks_dinov2"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

CKPT_PATH = f"{SAVE_DIR}/best_road_model_dinov2_fpn.pth"
MODEL_TAG = "dinov2_fpn"

# DINOv2 patch size is 14 → use 518 (14*37). Common HF / Meta default.
IMG_SIZE = 518
BATCH_SIZE = 2   # T4-safe; try 4 if VRAM allows
EPOCHS = 30
LR_BACKBONE = 1e-5
LR_DECODER = 1e-4
FREEZE_BACKBONE_EPOCHS = 3  # train FPN only for first N epochs

all_files = os.listdir(DATA_DIR)
sat = {f.replace("_sat.jpg", "") for f in all_files if f.endswith("_sat.jpg")}
mask = {f.replace("_mask.png", "") for f in all_files if f.endswith("_mask.png")}
all_ids = sorted(sat & mask)
random.seed(42)
random.shuffle(all_ids)
split = int(0.8 * len(all_ids))
train_ids, val_ids = all_ids[:split], all_ids[split:]
print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")
print(f"Checkpoint: {CKPT_PATH}")
print(f"IMG_SIZE={IMG_SIZE} (must be divisible by 14)")


## Dataset


In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_train_transforms(img_size=518):
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Resize(img_size, img_size),
        A.RandomShadow(
            shadow_roi=(0, 0, 1, 1),
            num_shadows_limit=(1, 3),
            shadow_intensity_range=(0.3, 0.7),
            p=0.5,
        ),
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(16, 48),
            hole_width_range=(16, 48),
            fill=0,
            p=0.5,
        ),
        A.RandomBrightnessContrast(p=0.4),
        A.HueSaturationValue(p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=518):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

class DeepGlobeDataset(Dataset):
    def __init__(self, data_dir, ids, transform=None):
        self.data_dir = data_dir
        self.ids = ids
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = cv2.cvtColor(
            cv2.imread(f"{self.data_dir}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB
        )
        mrgb = cv2.cvtColor(
            cv2.imread(f"{self.data_dir}/{img_id}_mask.png"), cv2.COLOR_BGR2RGB
        )
        mask = (mrgb[:, :, 0] > 127).astype(np.float32)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]
        return img, mask.unsqueeze(0)

print(f"dataset: {len(train_ids)} train, {len(val_ids)} val")


## Model

`facebook/dinov2-base` encoder + custom FPN decode head.

- DINOv2 = frozen / lightly fine-tuned feature extractor (robust under domain shift / occlusion)
- FPN = multi-scale fusion of intermediate ViT layers → binary road logits


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
import segmentation_models_pytorch as smp

class FPNDecoder(nn.Module):
    """Light FPN over 4 DINOv2 feature maps (same channel dim)."""

    def __init__(self, in_dim=768, fpn_dim=256, num_classes=1):
        super().__init__()
        self.laterals = nn.ModuleList([
            nn.Conv2d(in_dim, fpn_dim, kernel_size=1) for _ in range(4)
        ])
        self.smooth = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(fpn_dim, fpn_dim, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(fpn_dim),
                nn.ReLU(inplace=True),
            )
            for _ in range(4)
        ])
        self.head = nn.Sequential(
            nn.Conv2d(fpn_dim * 4, fpn_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(fpn_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(fpn_dim, num_classes, kernel_size=1),
        )

    def forward(self, feats, out_size):
        laterals = [lat(f) for lat, f in zip(self.laterals, feats)]
        for i in range(len(laterals) - 2, -1, -1):
            laterals[i] = laterals[i] + F.interpolate(
                laterals[i + 1], size=laterals[i].shape[-2:], mode="bilinear", align_corners=False
            )
        laterals = [s(f) for s, f in zip(self.smooth, laterals)]
        target = laterals[0].shape[-2:]
        fused = torch.cat(
            [F.interpolate(f, size=target, mode="bilinear", align_corners=False) for f in laterals],
            dim=1,
        )
        logits = self.head(fused)
        logits = F.interpolate(logits, size=out_size, mode="bilinear", align_corners=False)
        return logits


class Dinov2FPN(nn.Module):
    def __init__(self, backbone_name="facebook/dinov2-base", num_classes=1, freeze_backbone=False):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_name)
        self.hidden = self.backbone.config.hidden_size
        self.patch = self.backbone.config.patch_size
        n_layers = self.backbone.config.num_hidden_layers
        self.layer_ids = [n_layers // 4 - 1, n_layers // 2 - 1, (3 * n_layers) // 4 - 1, n_layers - 1]
        self.decoder = FPNDecoder(in_dim=self.hidden, fpn_dim=256, num_classes=num_classes)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def set_backbone_trainable(self, trainable: bool):
        for p in self.backbone.parameters():
            p.requires_grad = trainable

    def _tokens_to_map(self, tokens, h, w):
        x = tokens[:, 1:, :]
        b, n, c = x.shape
        gh, gw = h // self.patch, w // self.patch
        assert n == gh * gw, f"token count {n} != {gh}*{gw}"
        return x.reshape(b, gh, gw, c).permute(0, 3, 1, 2).contiguous()

    def forward(self, images):
        b, _, h, w = images.shape
        out = self.backbone(pixel_values=images, output_hidden_states=True)
        hs = out.hidden_states
        feats = [self._tokens_to_map(hs[i + 1], h, w) for i in self.layer_ids]
        return self.decoder(feats, out_size=(h, w))


def build_model(freeze_backbone=True):
    model = Dinov2FPN(freeze_backbone=freeze_backbone)
    n_all = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"DINOv2-B + FPN | total params: {n_all:,} | trainable: {n_train:,}")
    print(f"Using intermediate layers: {model.layer_ids}")
    return model


class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.4):
        super().__init__()
        self.alpha = alpha
        self.bce = smp.losses.SoftBCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)

    def forward(self, preds, targets):
        return self.alpha * self.bce(preds, targets) + (1 - self.alpha) * self.dice(preds, targets)


def calculate_metrics(preds, targets, threshold=0.5):
    preds = (torch.sigmoid(preds) > threshold).float()
    targets = targets.float()
    inter = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2.0 * inter + 1e-6) / (union + 1e-6)
    iou = (inter + 1e-6) / (union - inter + 1e-6)
    return iou.mean().item(), dice.mean().item()


## Training


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

train_loader = DataLoader(
    DeepGlobeDataset(DATA_DIR, train_ids, get_train_transforms(IMG_SIZE)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    DeepGlobeDataset(DATA_DIR, val_ids, get_val_transforms(IMG_SIZE)),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

model = build_model(freeze_backbone=True).to(device)
loss_fn = CombinedLoss(0.4)

def make_optimizer(model):
    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    decoder_params = list(model.decoder.parameters())
    groups = [{"params": decoder_params, "lr": LR_DECODER}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": LR_BACKBONE})
    return optim.AdamW(groups, weight_decay=0.05)

optimizer = make_optimizer(model)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

best_val_iou = 0.0
history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}
backbone_unfrozen = False

print(f"\nStarting training for {EPOCHS} epochs on {device}...")
print(f"Freeze backbone for first {FREEZE_BACKBONE_EPOCHS} epochs, then fine-tune at LR={LR_BACKBONE}\n")

for epoch in range(1, EPOCHS + 1):
    if (not backbone_unfrozen) and epoch > FREEZE_BACKBONE_EPOCHS:
        model.set_backbone_trainable(True)
        optimizer = make_optimizer(model)
        backbone_unfrozen = True
        print(f"Epoch {epoch}: unfroze DINOv2 backbone (differential LR)")

    model.train()
    t_loss = t_iou = 0.0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        images = images.to(device, torch.float32)
        masks = masks.to(device, torch.float32)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
        iou, _ = calculate_metrics(outputs, masks)
        t_iou += iou

    model.eval()
    v_loss = v_iou = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]"):
            images = images.to(device, torch.float32)
            masks = masks.to(device, torch.float32)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            v_loss += loss.item()
            iou, _ = calculate_metrics(outputs, masks)
            v_iou += iou

    t_loss /= len(train_loader)
    t_iou /= len(train_loader)
    v_loss /= len(val_loader)
    v_iou /= len(val_loader)
    scheduler.step(v_loss)

    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["train_iou"].append(t_iou)
    history["val_iou"].append(v_iou)

    print(
        f"Epoch {epoch:02d} | Train Loss: {t_loss:.4f} IoU: {t_iou:.4f} | "
        f"Val Loss: {v_loss:.4f} IoU: {v_iou:.4f}"
    )
    if v_iou > best_val_iou:
        best_val_iou = v_iou
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"  new best val iou {v_iou:.4f}")

print(f"\nfinished. best val iou {best_val_iou:.4f}")
print(f"Checkpoint: {CKPT_PATH}")


## Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train Loss", color="tomato")
axes[0].plot(history["val_loss"], label="Val Loss", color="steelblue")
axes[0].set_title("Loss Curves")
axes[0].legend(); axes[0].grid(True)
axes[1].plot(history["train_iou"], label="Train IoU", color="tomato")
axes[1].plot(history["val_iou"], label="Val IoU", color="steelblue")
axes[1].set_title("IoU Score Curves")
axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves_{MODEL_TAG}.png", dpi=120)
plt.show()
print(f"Best Val IoU: {best_val_iou:.4f}")


## Inference


In [ ]:
model = build_model(freeze_backbone=False).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model = model.eval()
print("Loaded", CKPT_PATH)

@torch.no_grad()
def predict_and_save(img_id):
    img = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    aug = get_val_transforms(IMG_SIZE)(image=img, mask=np.zeros(img.shape[:2], dtype=np.float32))
    t = aug["image"].unsqueeze(0).to(device)
    logits = model(t)
    pred = (torch.sigmoid(logits) > 0.5).squeeze().cpu().numpy().astype(np.uint8) * 255
    if pred.shape != (h, w):
        pred = cv2.resize(pred, (w, h), interpolation=cv2.INTER_NEAREST)
    out_path = f"{MASK_DIR}/{img_id}_pred.png"
    cv2.imwrite(out_path, pred)
    return pred

DEMO_IDS = ["493626", "477671", "422265", "194764"]
demo_ids = [i for i in DEMO_IDS if i in val_ids or i in train_ids]
if not demo_ids:
    demo_ids = val_ids[:4]

for img_id in demo_ids:
    predict_and_save(img_id)
    print(f"Saved mask: {MASK_DIR}/{img_id}_pred.png")

print("Done. Masks in", MASK_DIR)
print("For Phase II: copy masks_dinov2 → masks_deeplab or change MASK_DIR in phase2 notebook.")


## Quick visual check


In [ ]:
sample_ids = demo_ids[:4]
fig, axes = plt.subplots(len(sample_ids), 3, figsize=(12, 4 * len(sample_ids)))
if len(sample_ids) == 1:
    axes = axes.reshape(1, -1)
for row, img_id in enumerate(sample_ids):
    sat = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB)
    gt = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_mask.png"), cv2.COLOR_BGR2RGB)
    pred = cv2.imread(f"{MASK_DIR}/{img_id}_pred.png", cv2.IMREAD_GRAYSCALE)
    axes[row, 0].imshow(sat); axes[row, 0].set_title(f"{img_id} satellite"); axes[row, 0].axis("off")
    axes[row, 1].imshow(gt); axes[row, 1].set_title("Ground truth"); axes[row, 1].axis("off")
    axes[row, 2].imshow(pred, cmap="gray"); axes[row, 2].set_title("Prediction"); axes[row, 2].axis("off")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/predictions_{MODEL_TAG}.png", dpi=120)
plt.show()
